# Lab 01 — 時間序列特徵工程

**Pipeline 中的角色：** 把原始採集指標轉換成可偵測的特徵向量。
這是整套 AIOps pipeline 的第二個環節，也是決定後面所有偵測層「能看到什麼」的關鍵一步。

原始的 `node_network_receive_bytes_total` 是個只增不減的 Counter——它不直接告訴你「current traffic高不高」。
你需要 `rate()` 換算成速率、滾動視窗建立動態基線、組合多個指標萃取語意——
這些轉換步驟加起來，就是特徵工程。

沒有好的特徵，後面三個偵測層（統計基線、SPC、ML）都是對著噪音做計算。


## Lab 總覽

### 學習目標
1. 理解 Counter and Rate 的差異，以及 rolling window（滑動視窗）的直觀含義
2. 從 Prometheus 拉取真實網路流量並計算複合特徵（流量、錯誤率、不對稱性）
3. 掌握滾動統計（Rolling Statistics）、Lag 特徵、多解析度重採樣
4. 理解「特徵選擇是人類決策」的核心概念

### 前置條件
- 已完成 Lab 00（Prometheus 服務正常或已接受 fallback 合成資料）
- `pandas`、`matplotlib`、`numpy` 已安裝

## 設計主線：把原始指標翻譯成可偵測訊號

本章是整個 workshop 的演算法入口。你不是在「整理資料」而已，而是在決定後面的偵測器能不能看見流量尖峰、錯誤率、非對稱傳輸或週期性模式。

請用三個實務問題檢查 feature design：

1. **這個 feature 對應哪個故障模式**：traffic surge、link degradation、broadcast storm、capacity pressure 是否都被表示出來？
2. **這個 feature 是否能跨介面比較**：大 port 和小 port 是否需要 ratio、z-score 或 per-port baseline？
3. **這個 feature 是否支援下一章告警**：如果 feature 沒有穩定 baseline，Lab 02 的 threshold 會很難校準。

設計原則：特徵是營運語意的編碼。演算法品質先受 feature design 限制。


## 本章在完整 Pipeline 的位置

圖表說明：本章（Lab 01）接收 Lab 00 採集的原始 Counter 指標，
輸出三類特徵（速率 / 複合比例 / time features）供 Labs 02–04 偵測層使用。

本章圖表來源為 repository 內的 SVG：`labs/self-study/diagrams/lab01_feature_pipeline.svg`。


In [ ]:
from pathlib import Path
from IPython.display import SVG

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "environment.yml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SVG(filename=str(PROJECT_ROOT / "labs/self-study/diagrams/lab01_feature_pipeline.svg"))
(PROJECT_ROOT / "outputs" / "workshop").mkdir(parents=True, exist_ok=True)

---
**概念說明 ▸ 講師引導** — 請聆聽講師說明，完成後再執行以下 cell。

---

## 概念：Counter vs Rate，以及 視窗 作為「移動記憶」

### Counter vs Rate

```
node_network_receive_bytes_total (Counter)

  value ↑  monotonically increasing
     (B)│              ┌─────────── resets to 0 on restart; rate() compensates automatically
        │   ───────────┘
        └───────────────────────────→ Time

  "Total bytes received since node_exporter started"


rate(node_network_receive_bytes_total[1m])  (bytes/sec rate)

  value ↑
(B/s) │     ╱╲           ╱╲
      │    ╱  ╲         ╱  ╲    ← reflects real traffic fluctuation
      │───╱    ╲───────╱    ╲───
      └───────────────────────→ Time

  "Average bytes received per second over the past 1 minute"
```

**公式（rate() 的計算方式）：**

```
rate(metric[w]) = (value at window end − value at window start) ÷ window seconds

Example ([1m] window):
  t=0s : bytes_total = 1,000,000
  t=60s: bytes_total = 1,060,000

  rate = (1,060,000 − 1,000,000) / 60 = 1,000 bytes/sec = 1 KB/s
```

---

### rolling window：窗口是移動的記憶

```
Time series (one sample per minute):

  t=1  t=2  t=3  t=4  t=5  t=6  t=7  t=8  t=9  t=10
  [10] [12] [11] [15] [13] [14] [12] [11] [16] [14]

  Window size = 3 (remembers the past 3 samples):

  t=3: window = [10, 12, 11]  → mean = 11.0, std = 1.0
  t=4: window = [12, 11, 15]  → mean = 12.7, std = 2.1  ← window shifts right
  t=5: window = [11, 15, 13]  → mean = 13.0, std = 2.0
              ─────────────────────
              The window is a "moving memory frame" tracking local statistics
```

**為什麼需要移動窗口，而不是全局統計？**

網路流量有日夜週期性：白天 10 MB/s 是正常，凌晨 10 MB/s 是異常。
用全局 mean 當基線 → 白天永遠告警、深夜永遠正常 → 基線沒有意義。
用滾動 mean（局部統計）→ 基線追蹤「最近的正常」→ 偵測「偏離最近正常」。

---

### 直覺：特徵工程是「翻譯」

```
Raw metrics (machine language)   Features (analysis language)

bytes_total (Counter)   →   rate: bytes/sec
rate_rx + rate_tx       →   traffic_total (total throughput)
errors / packets        →   error_rate (quality indicator)
rolling(mean, 12min)    →   baseline (dynamic baseline)
value - rolling_mean    →   deviation
deviation / rolling_std →   z_score (normalized deviation)
```

每一步翻譯都讓資料更「可比較」——不同設備、不同時段的數字放在同一把尺上衡量。
這是特徵工程的核心目的，也是為什麼 Lab 01 是整個 pipeline 的第二關。


---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 0：環境準備

In [ ]:
import urllib.request
import urllib.parse
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 11

PROM = "http://localhost:9090"

def prom_range(expr, hours=6, step="15s"):
    end = datetime.utcnow()
    start = end - timedelta(hours=hours)
    params = urllib.parse.urlencode({
        "query": expr,
        "start": start.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "end":   end.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "step":  step,
    })
    with urllib.request.urlopen(f"{PROM}/api/v1/query_range?{params}", timeout=8) as r:
        return json.loads(r.read())

def to_df(result, value_col="value"):
    rows = []
    for series in result["data"]["result"]:
        dev = series["metric"].get("device",
              series["metric"].get("instance", "?"))
        for ts, val in series["values"]:
            rows.append({"timestamp": pd.Timestamp(ts, unit="s"),
                          "device": dev, value_col: float(val)})
    return pd.DataFrame(rows)

print("✅ 環境準備完成")

---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 1：從 Prometheus 拉取 6 小時速率資料

In [ ]:
def generate_synthetic_6h():
    """合成 6 小時的多指標網路資料（Prometheus 不可用時的 fallback）。"""
    np.random.seed(42)
    n = 360  # 6h * 60min
    end = datetime.utcnow()
    timestamps = [pd.Timestamp(end - timedelta(minutes=n-i)) for i in range(n)]
    
    # 模擬日間Traffic模式（下午較高）
    t = np.linspace(0, 2 * np.pi, n)
    base_rx = 2_000_000 + 1_500_000 * np.sin(t * 0.3)  # 2-3.5 MB/s
    base_tx = 500_000 + 300_000 * np.sin(t * 0.3 + 0.5)
    
    noise_rx = np.random.normal(0, 200_000, n)
    noise_tx = np.random.normal(0, 50_000, n)
    
    rx = np.maximum(base_rx + noise_rx, 0)
    tx = np.maximum(base_tx + noise_tx, 0)
    
    # 加入突刺（模擬異常）
    for spike_i in [90, 180, 270]:
        rx[spike_i:spike_i+2] *= 8
        tx[spike_i:spike_i+2] *= 3
    
    pkts = rx / 1500 + np.random.normal(0, 50, n)  # 模擬封包數
    errs = np.maximum(np.random.exponential(0.01, n), 0)  # 錯誤率（通常很低）
    drops = np.maximum(np.random.exponential(0.005, n), 0)
    
    df = pd.DataFrame({
        'timestamp': timestamps,
        'device': 'eth0',
        'rx_rate': rx,
        'tx_rate': tx,
        'rx_packets': pkts,
        'tx_packets': pkts * 0.3,
        'rx_errors': errs,
        'tx_errors': errs * 0.5,
        'rx_drops': drops,
        'tx_drops': drops * 0.2,
    })
    return df

filter_expr = 'device!~"lo|docker.*|veth.*|br-.*"'

# 嘗試從 Prometheus 拉取多個指標
metrics = {
    'rx_rate':   f'rate(node_network_receive_bytes_total{{{filter_expr}}}[1m])',
    'tx_rate':   f'rate(node_network_transmit_bytes_total{{{filter_expr}}}[1m])',
    'rx_packets':f'rate(node_network_receive_packets_total{{{filter_expr}}}[1m])',
    'tx_packets':f'rate(node_network_transmit_packets_total{{{filter_expr}}}[1m])',
    'rx_errors': f'rate(node_network_receive_errs_total{{{filter_expr}}}[1m])',
    'tx_errors': f'rate(node_network_transmit_errs_total{{{filter_expr}}}[1m])',
    'rx_drops':  f'rate(node_network_receive_drop_total{{{filter_expr}}}[1m])',
    'tx_drops':  f'rate(node_network_transmit_drop_total{{{filter_expr}}}[1m])',
}

try:
    dfs = {}
    for col, expr in metrics.items():
        r = prom_range(expr, hours=6, step="1m")
        dfs[col] = to_df(r, col)
    
    # 以 rx_rate 為基礎，merge 所有指標
    df_raw = dfs['rx_rate'].copy()
    for col, dfc in dfs.items():
        if col == 'rx_rate': continue
        merged = dfc[['timestamp', 'device', col]]
        df_raw = df_raw.merge(merged, on=['timestamp', 'device'], how='left')
    
    df_raw = df_raw.fillna(0).sort_values(['device', 'timestamp']).reset_index(drop=True)
    print(f"✅ Prometheus：取得 {len(df_raw)} 筆資料")
    print(f"   裝置：{df_raw['device'].unique().tolist()}")
    print(f"   欄位：{list(df_raw.columns)}")

except Exception as e:
    print(f"⚠ Prometheus 無法連線（{e}）— 使用合成 6h 資料")
    df_raw = generate_synthetic_6h()
    print(f"   合成資料：{len(df_raw)} 筆，欄位：{list(df_raw.columns)}")

df_raw.head(3)

In [ ]:
# 選取主要裝置，確保時序連續
DEVICE = df_raw['device'].value_counts().index[0]  # 資料量最多的裝置
df = df_raw[df_raw['device'] == DEVICE].copy().sort_values('timestamp').reset_index(drop=True)

print(f"主要分析裝置：{DEVICE}")
print(f"資料筆數：{len(df)}")
print(f"時間範圍：{df['timestamp'].iloc[0]} ~ {df['timestamp'].iloc[-1]}")

# 快速視覺化原始資料
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(df['timestamp'], df['rx_rate'] / 1e6, color='steelblue', linewidth=1, label='Receive')
axes[0].plot(df['timestamp'], df['tx_rate'] / 1e6, color='tomato', linewidth=1, label='Transmit')
axes[0].set_ylabel('Rate (MB/s)')
axes[0].set_title(f'{DEVICE} — 6-hour network traffic')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(df['timestamp'], df['rx_errors'], color='orange', linewidth=1, label='RX errors')
axes[1].plot(df['timestamp'], df['rx_drops'], color='red', linewidth=1, label='RX drops')
axes[1].set_ylabel('errors/s')
axes[1].set_title('Receive errors and drops')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

plt.tight_layout()
plt.show()

---

## EDA：探索性資料分析

在設計任何特徵之前，先花 5 分鐘看清楚資料的本質。
EDA 回答三個問題，每個問題都直接影響後面的特徵工程決策：

| 問題 | 工具 | 影響的決策 |
|------|------|-----------|
| 資料品質好嗎？ | 缺值、重複、採樣間隔 | 要不要補值？用哪種補值策略？ |
| 分布長什麼形狀？ | 直方圖、偏態、IQR | 要不要做 log 轉換？閾值設多少？ |
| 有時間週期性嗎？ | 自相關、ACF | lag 值要設多少？WINDOW 的上下限是多少？ |

跳過 EDA 的代價：選錯 WINDOW、lag 特徵沒用、閾值設在正常值中間、Z-score 的分布假設錯誤。


In [ ]:
# ── EDA Step A：資料品質檢查 ──────────────────────────────────────────────────
import numpy as np

print("▶ 基本結構")
print(f"  資料筆數  ：{len(df):,}")
print(f"  欄位數量  ：{len(df.columns)}")
print(f"  時間範圍  ：{df['timestamp'].iloc[0].strftime('%H:%M')} ~ {df['timestamp'].iloc[-1].strftime('%H:%M')}")
total_minutes = (df['timestamp'].iloc[-1] - df['timestamp'].iloc[0]).total_seconds() / 60
print(f"  總時長    ：{total_minutes:.0f} 分鐘（{total_minutes/60:.1f} 小時）")
expected_n = int(total_minutes) + 1
print(f"  預期筆數  ：{expected_n}（每分鐘 1 筆）")
print(f"  實際筆數  ：{len(df)}  {'✅ 完整' if len(df) >= expected_n * 0.99 else '⚠️ 可能有缺漏'}")

print()
print("▶ 缺值統計")
null_counts = df.isnull().sum()
if null_counts.sum() == 0:
    print("  ✅ 無缺值")
else:
    for col, cnt in null_counts[null_counts > 0].items():
        print(f"  {col:<20} {cnt} 筆 ({100*cnt/len(df):.1f}%)")

print()
print("▶ 採樣間隔分析")
diffs = df['timestamp'].diff().dt.total_seconds().dropna()
print(f"  中位數間隔：{diffs.median():.0f} 秒")
print(f"  最小間隔  ：{diffs.min():.0f} 秒")
print(f"  最大間隔  ：{diffs.max():.0f} 秒")
irregular = int((diffs != diffs.median()).sum())
print(f"  不規則間隔：{irregular} 筆 {'✅ 均勻採樣' if irregular == 0 else '⚠️ 採樣不均，預處理需重採樣'}")

print()
print("▶ 重複時間戳記")
dup = int(df['timestamp'].duplicated().sum())
print(f"  重複筆數  ：{dup} {'✅' if dup == 0 else '⚠️ 需去除重複'}")

print()
print("▶ 零值與負值（不合理觀測）")
numeric_cols = ['rx_rate','tx_rate','rx_packets','tx_packets','rx_errors','tx_errors','rx_drops','tx_drops']
for col in numeric_cols:
    if col not in df.columns: continue
    neg = int((df[col] < 0).sum())
    zeros = int((df[col] == 0).sum())
    if neg > 0 or zeros > len(df) * 0.1:
        print(f"  {col:<20} 負值={neg}  零值={zeros} ({100*zeros/len(df):.1f}%)")
print("  （無顯示 = 所有欄位均無負值，且零值 < 10%）")


In [ ]:
# ── EDA Step B：分布形狀分析 ──────────────────────────────────────────────────
from scipy import stats as _scipy_stats

target_cols = ['rx_rate', 'tx_rate', 'traffic_total']
target_cols = [c for c in target_cols if c in df.columns]

fig, axes = plt.subplots(2, len(target_cols), figsize=(5*len(target_cols), 7))
if len(target_cols) == 1: axes = axes.reshape(2, 1)

print("▶ 分布統計量")
print(f"  {'欄位':<20} {'均值 MB/s':>10} {'中位 MB/s':>10} {'偏態':>8} {'峰態':>8}  判斷")
print("  " + "─"*70)
for j, col in enumerate(target_cols):
    s = df[col].dropna() / 1e6
    skew = float(_scipy_stats.skew(s))
    kurt = float(_scipy_stats.kurtosis(s))
    if abs(skew) < 0.5:
        verdict = "✅ 近似常態"
    elif skew > 1.5:
        verdict = "⚠️ 右偏重尾（考慮 log 轉換）"
    elif skew < -1.5:
        verdict = "⚠️ 左偏（少見，檢查資料源）"
    else:
        verdict = "⚠️ 輕微偏態（Z-score 仍可用）"
    print(f"  {col:<20} {s.mean():>10.3f} {s.median():>10.3f} {skew:>8.3f} {kurt:>8.3f}  {verdict}")

    # Top row: histogram + KDE
    ax_hist = axes[0][j]
    ax_hist.hist(s, bins=40, color='steelblue', alpha=0.7, density=True, label='Observed distribution')
    xrange = np.linspace(s.min(), s.max(), 200)
    mu, sigma = s.mean(), s.std()
    ax_hist.plot(xrange, _scipy_stats.norm.pdf(xrange, mu, sigma),
                 'r--', linewidth=1.5, label='Normal reference')
    ax_hist.set_title(f'{col}\nskew={skew:.2f} kurtosis={kurt:.2f}', fontsize=9)
    ax_hist.set_xlabel('MB/s', fontsize=8)
    ax_hist.legend(fontsize=7)

    # Bottom row: boxplot with outlier annotation
    ax_box = axes[1][j]
    bp = ax_box.boxplot(s, vert=True, patch_artist=True,
                        boxprops=dict(facecolor='lightsteelblue', alpha=0.7),
                        flierprops=dict(marker='o', color='red', markersize=3, alpha=0.5))
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    outlier_cnt = int(((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum())
    ax_box.set_title(f'IQR outliers: {outlier_cnt} points', fontsize=9)
    ax_box.set_ylabel('MB/s', fontsize=8)
    ax_box.set_xticks([])

plt.suptitle('Distribution shape analysis (histogram vs normal reference + boxplot outliers)', fontsize=12, y=1.01)
plt.tight_layout();  plt.show()


In [ ]:
# ── EDA Step C：自相關分析（決定 lag 和 WINDOW 的客觀依據）──────────────────
import matplotlib.gridspec as _gs

TARGET_COL_ACF = 'rx_rate' if 'rx_rate' in df.columns else df.select_dtypes('number').columns[0]
series_acf = df[TARGET_COL_ACF].dropna()
MAX_LAG = min(60, len(series_acf) // 3)

# ── 計算 ACF ─────────────────────────────────────────────────────────────────
acf_vals = [series_acf.autocorr(lag=k) for k in range(0, MAX_LAG + 1)]
lags = list(range(0, MAX_LAG + 1))
conf95 = 1.96 / np.sqrt(len(series_acf))  # 95% 信賴區間

# ── 找顯著 lag ───────────────────────────────────────────────────────────────
sig_lags = [(k, v) for k, v in zip(lags[1:], acf_vals[1:]) if abs(v) > conf95]

fig = plt.figure(figsize=(12, 5))
spec = _gs.GridSpec(1, 2, width_ratios=[3, 1], figure=fig)

# 左：ACF 棒狀圖
ax_acf = fig.add_subplot(spec[0])
ax_acf.bar(lags, acf_vals, color=['steelblue' if abs(v) > conf95 else 'lightgray'
                                   for v in acf_vals], width=0.7)
ax_acf.axhline( conf95, color='red', linestyle='--', linewidth=0.8, label='±95% CI')
ax_acf.axhline(-conf95, color='red', linestyle='--', linewidth=0.8)
ax_acf.axhline(0, color='black', linewidth=0.5)
ax_acf.set_xlabel('Lag (minutes)');  ax_acf.set_ylabel('Autocorrelation r')
ax_acf.set_title(f'{TARGET_COL_ACF} autocorrelation function (ACF)')
ax_acf.legend(fontsize=8)

# 右：Top-10 顯著 lag 排行
ax_top = fig.add_subplot(spec[1])
top_sig = sorted(sig_lags, key=lambda x: -abs(x[1]))[:10]
if top_sig:
    top_lags_v, top_r_v = zip(*top_sig)
    colors_bar = ['coral' if v > 0 else 'steelblue' for v in top_r_v]
    ax_top.barh([f'lag_{k}' for k in top_lags_v], [abs(v) for v in top_r_v],
                color=colors_bar, alpha=0.8)
    ax_top.set_xlabel('|r|');  ax_top.set_title('Top significant lags')
    ax_top.axvline(conf95, color='red', linestyle='--', linewidth=0.8)

plt.tight_layout();  plt.show()

# ── 解讀 ──────────────────────────────────────────────────────────────────────
print(f"\n▶ ACF 解讀（{TARGET_COL_ACF}，95% CI = ±{conf95:.3f}）")
print(f"  顯著 lag 數量：{len(sig_lags)} / {MAX_LAG}")
print()
if not sig_lags:
    print("  ❌ 無顯著自相關 → 序列接近白噪音，lag 特徵沒有預測價值")
else:
    best_lag, best_r = max(sig_lags, key=lambda x: abs(x[1]))
    print(f"  最強自相關：lag_{best_lag} (r = {best_r:.4f})")
    print()
    print("  → 特徵工程建議（對應 Step 4）：")
    for k, v in sorted(sig_lags[:6], key=lambda x: x[0]):
        if abs(v) >= 0.7:
            quality = "強，值得作特徵"
        elif abs(v) >= 0.4:
            quality = "中等，可以嘗試"
        else:
            quality = "弱，謹慎評估"
        print(f"    lag_{k:>3} min  r={v:>7.4f}  {quality}")
    print()
    # Suggest WINDOW range
    # WINDOW lower bound: should be > longest significant lag (to avoid baseline being pulled by anomaly)
    max_sig_lag = max(sig_lags, key=lambda x: x[0])[0]
    print(f"  → WINDOW 建議（對應 Step 3）：")
    print(f"    最長顯著 lag = {max_sig_lag} 分鐘")
    print(f"    WINDOW 建議下限 ≈ {max_sig_lag} 分鐘（基線需要覆蓋一個完整週期）")
    print(f"    WINDOW 建議上限 ≈ {max_sig_lag * 4} 分鐘（太長會讓基線追不上漸進式異常）")


In [ ]:
# ── EDA Step D：離群值偵測與異常候選期 ───────────────────────────────────────
TARGET_COL_OUT = 'rx_rate' if 'rx_rate' in df.columns else df.select_dtypes('number').columns[0]
s = df[TARGET_COL_OUT].copy()

# IQR 方法
q1, q3 = s.quantile(0.25), s.quantile(0.75)
iqr = q3 - q1
lower_iqr = q1 - 1.5 * iqr
upper_iqr = q3 + 1.5 * iqr

# 3-sigma 方法
mu, sigma = s.mean(), s.std()
lower_3s = mu - 3 * sigma
upper_3s = mu + 3 * sigma

flag_iqr = (s < lower_iqr) | (s > upper_iqr)
flag_3s  = (s < lower_3s)  | (s > upper_3s)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# 上圖：時序 + 異常標記
axes[0].plot(df['timestamp'], s / 1e6, linewidth=0.8, color='steelblue', alpha=0.8, label=TARGET_COL_OUT)
axes[0].axhline(upper_iqr / 1e6, color='orange', linestyle='--', linewidth=1, label=f'IQR upper = {upper_iqr/1e6:.2f} MB/s')
axes[0].axhline(upper_3s  / 1e6, color='red',    linestyle='--', linewidth=1, label=f'3σ upper = {upper_3s/1e6:.2f} MB/s')
iqr_hits = df['timestamp'][flag_iqr]
s3_hits  = df['timestamp'][flag_3s]
axes[0].scatter(iqr_hits, s[flag_iqr] / 1e6, color='orange', s=18, zorder=5, label=f'IQR outliers ({int(flag_iqr.sum())})')
axes[0].scatter(s3_hits,  s[flag_3s]  / 1e6, color='red',    s=30, zorder=6, marker='^', label=f'3σ outliers ({int(flag_3s.sum())})')
axes[0].set_ylabel('MB/s');  axes[0].legend(fontsize=8)
axes[0].set_title(f'{TARGET_COL_OUT} — outlier locations over time')
axes[0].grid(True, alpha=0.3)

# 下圖：各時段 20-min rolling 最大值 heatmap 概念
rolling_max = s.rolling(20, center=True).max() / 1e6
axes[1].fill_between(df['timestamp'], rolling_max, alpha=0.4, color='coral', label='20-min rolling max')
axes[1].plot(df['timestamp'], s / 1e6, linewidth=0.5, color='steelblue', alpha=0.5)
axes[1].set_ylabel('MB/s');  axes[1].legend(fontsize=8)
axes[1].set_title('20-minute rolling maximum (peak periods are visible)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout();  plt.show()

# ── 解讀 ─────────────────────────────────────────────────────────────────────
print(f"\n▶ 離群值解讀（{TARGET_COL_OUT}）")
print(f"  IQR 方法：{int(flag_iqr.sum())} 筆 ({100*flag_iqr.mean():.2f}%)   "
      f"上界 = {upper_iqr/1e6:.3f} MB/s  下界 = {lower_iqr/1e6:.3f} MB/s")
print(f"  3σ  方法：{int(flag_3s.sum())} 筆 ({100*flag_3s.mean():.2f}%)   "
      f"上界 = {upper_3s/1e6:.3f} MB/s  下界 = {lower_3s/1e6:.3f} MB/s")
print()
print("  兩種方法的差異意義：")
print("  ┌────────────┬──────────────────────────────────────────────┐")
print("  │ IQR        │ 不假設常態分布，對重尾序列更穩健              │")
print("  │ 3σ（Z-score）│ 假設常態分布，極端值定義更嚴格               │")
print("  └────────────┴──────────────────────────────────────────────┘")
print()
iqr_unique = int((flag_iqr & ~flag_3s).sum())
s3_unique  = int((flag_3s & ~flag_iqr).sum())
both       = int((flag_iqr & flag_3s).sum())
print(f"  僅 IQR 捕捉（輕微離群）：{iqr_unique} 筆  → 可能是正常的流量高峰")
print(f"  僅 3σ 捕捉              ：{s3_unique} 筆")
print(f"  兩者都捕捉（極端離群）  ：{both} 筆  → 高概率異常，預處理需處理")
print()
print("  → 預處理建議（下方 Preprocessing 步驟）：")
if both > 0:
    print(f"    1. 為 {both} 個兩方法都捕捉的極端值加上 flag_outlier 欄位（不刪除，標記）")
if int(flag_iqr.sum()) / max(len(s), 1) > 0.05:
    print(f"    2. IQR 離群比例 {100*flag_iqr.mean():.1f}% > 5%，建議 Cap 而非刪除（保留時序連續性）")
else:
    print(f"    2. IQR 離群比例 {100*flag_iqr.mean():.1f}% 正常，可選擇保留原始值或輕微 cap")


---

## EDA 結論與行動清單

執行完上面四個 EDA cell 後，把你的觀察填在下面三格：

---

### 🔎 資料品質觀察

> 執行 EDA Step A 後填寫：

| 問題 | 你的觀察 | 預處理動作 |
|------|---------|-----------|
| 有缺值嗎？ | 例：rx_drops 有 5 筆 NaN | 前向填補（forward fill）|
| 採樣是否均勻？ | 例：全部 60 秒間隔，✅ 均勻 | 不需重採樣 |
| 有重複時間戳記？ | 例：無重複 ✅ | — |
| 有不合理的負值？ | 例：無 ✅ | — |

---

### 📊 分布形狀觀察（影響 Z-score 和 log 轉換決策）

> 執行 EDA Step B 後填寫：

| 欄位 | 偏態值 | 分布判斷 | 建議 |
|------|--------|---------|------|
| rx_rate | 例：2.3 | 右偏重尾 | 考慮 log 轉換後再做 Z-score |
| tx_rate | | | |
| error_rate | | | |

**右偏重尾的影響：**
Z-score 假設常態分布。右偏序列的「正常波動」會讓 Z-score 的標準差估計偏大，
真實異常的 Z-score 反而沒有想像中高，靈敏度下降。
解法：`df['rx_rate_log'] = np.log1p(df['rx_rate'])`，在 log 域做 Z-score。

---

### 📈 自相關觀察（決定 lag 和 WINDOW 的客觀依據）

> 執行 EDA Step C 後填寫：

| 結論 | 你的數值 | 對應決策 |
|------|---------|---------|
| 最強自相關 lag | 例：lag_12，r = 0.87 | Step 4 的 LAGS 必選 12 |
| 第二強 lag | 例：lag_1，r = 0.92 | 加入 LAGS = [1, 12, ...] |
| 建議 WINDOW 下限 | 例：12 分鐘 | Step 3 的 WINDOW ≥ 12 |
| 建議 WINDOW 上限 | 例：48 分鐘 | Step 3 的 WINDOW ≤ 48 |

---

### ⚠️ 異常候選期（供 Lab 02 和 Lab 08 參考）

> 執行 EDA Step D 後填寫：

| 時間窗口 | 現象 | 可能原因 |
|---------|------|---------|
| 例：10:30–10:35 | rx_rate > 3σ 上界，持續 5 分鐘 | 排程備份 / DDoS / 設備故障 |
| | | |


---

## 資料預處理

EDA 找到的問題，在這裡統一處理。
預處理的三個原則：

1. **不刪除，只標記** — 時序資料刪點會破壞連續性，改用 `flag_outlier` 欄位
2. **補值選策略** — rate 類欄位用前向填補（趨勢延續），count 類欄位用 0（沒有封包 = 0）
3. **先處理再特徵工程** — 所有 Step 2–5 的特徵都在乾淨的 `df` 上計算



In [ ]:
# ── 預處理 Step 1：補缺值 ────────────────────────────────────────────────────
print("▶ 補值前缺值數量：")
print(df.isnull().sum()[df.isnull().sum() > 0].to_string() or "  （無缺值）")

# Rate 類欄位：前向填補（假設上一分鐘的速率延續）
rate_cols  = [c for c in ['rx_rate','tx_rate'] if c in df.columns]
# Count 類欄位：填 0（沒有封包 = 0 更合理）
count_cols = [c for c in ['rx_packets','tx_packets','rx_errors','tx_errors','rx_drops','tx_drops']
              if c in df.columns]

df[rate_cols]  = df[rate_cols].ffill().bfill()
df[count_cols] = df[count_cols].fillna(0)

print("▶ 補值後缺值數量：")
remaining = df.isnull().sum().sum()
print(f"  剩餘缺值：{remaining} {'✅' if remaining == 0 else '⚠️ 仍有缺值，請檢查'}")

# ── 預處理 Step 2：確保 1 分鐘等間隔索引 ──────────────────────────────────────
print()
print("▶ 等間隔重採樣")
diffs = df['timestamp'].diff().dt.total_seconds().dropna()
if diffs.std() > 5:    # 間隔標準差 > 5 秒才重採樣
    df_indexed = df.set_index('timestamp').resample('1min').mean()
    df_indexed = df_indexed.interpolate(method='time')    # 線性插值填補重採樣後的 NaN
    df = df_indexed.reset_index()
    print(f"  已重採樣為 1min 間隔，現有 {len(df)} 筆")
else:
    print(f"  ✅ 採樣間隔均勻（std = {diffs.std():.1f}s），不需重採樣")

# ── 預處理 Step 3：Cap 極端離群值 + 加標記欄位 ────────────────────────────────
print()
print("▶ 離群值處理（IQR + 3σ 雙重確認）")
cap_col = 'rx_rate'
if cap_col in df.columns:
    q1, q3 = df[cap_col].quantile(0.25), df[cap_col].quantile(0.75)
    iqr = q3 - q1
    mu, sigma = df[cap_col].mean(), df[cap_col].std()
    cap_upper_iqr = q3 + 1.5 * iqr
    cap_upper_3s  = mu + 3 * sigma

    # 同時超過兩種閾值才認定為「需處理的極端值」
    extreme_mask = (df[cap_col] > cap_upper_iqr) & (df[cap_col] > cap_upper_3s)
    df['flag_outlier'] = extreme_mask.astype(int)

    extreme_count = int(extreme_mask.sum())
    if extreme_count > 0:
        # Cap 到 3σ 上界（保留相對順序，不刪除）
        cap_value = cap_upper_3s
        df.loc[extreme_mask, cap_col] = cap_value
        print(f"  {cap_col}：{extreme_count} 個極端值 cap 至 {cap_value/1e6:.3f} MB/s（3σ 上界）")
        print(f"  flag_outlier 欄位標記這些時間點，Lab 08 RCA 可使用")
    else:
        df['flag_outlier'] = 0
        print(f"  {cap_col}：無需 cap 的極端值 ✅")


In [ ]:
# ── 預處理 Step 4：驗證與預處理後概覽 ───────────────────────────────────────
print("▶ 預處理後資料狀態")
print(f"  筆數          ：{len(df)}")
print(f"  缺值總計      ：{df.isnull().sum().sum()} {'✅' if df.isnull().sum().sum() == 0 else '⚠️'}")
print(f"  flag_outlier  ：{int(df['flag_outlier'].sum())} 筆被標記")

# 採樣間隔確認
diffs_post = df['timestamp'].diff().dt.total_seconds().dropna()
print(f"  採樣間隔      ：{diffs_post.median():.0f}s（std={diffs_post.std():.2f}s）")

# 數值範圍確認（應無負值）
neg_check = (df[['rx_rate','tx_rate']].values < 0).sum()
print(f"  負值（rx/tx） ：{neg_check} {'✅' if neg_check == 0 else '⚠️'}")

# 視覺化對比：預處理前後
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

axes[0].plot(df['timestamp'], df['rx_rate'] / 1e6,
             linewidth=0.8, color='steelblue', label='rx_rate (preprocessed)')
if 'flag_outlier' in df.columns:
    flagged = df[df['flag_outlier'] == 1]
    axes[0].scatter(flagged['timestamp'], flagged['rx_rate'] / 1e6,
                    color='red', s=25, zorder=5, label=f'capped extreme values ({len(flagged)})')
axes[0].set_ylabel('MB/s');  axes[0].legend(fontsize=8)
axes[0].set_title('rx_rate - preprocessed (red points = capped extremes)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df['timestamp'], df['tx_rate'] / 1e6,
             linewidth=0.8, color='coral', label='tx_rate')
axes[1].set_ylabel('MB/s');  axes[1].legend(fontsize=8)
axes[1].set_title('tx_rate - preprocessed')
axes[1].grid(True, alpha=0.3)

plt.tight_layout();  plt.show()

print()
print("✅ 預處理完成，df 可進入 Step 2 特徵工程")
print(f"   欄位清單：{list(df.columns)}")


---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 2：計算複合特徵

三個衍生特徵各自捕捉不同的異常信號：

| 特徵 | 公式 | 捕捉什麼異常 |
|------|------|-------------|
| `traffic_total` | rx_rate + tx_rate | 總流量突增 / 驟降 |
| `error_rate` | (rx_errs + tx_errs) / (rx_pkts + tx_pkts) | 網路品質惡化（封包損失）|
| `tx_ratio` | tx_rate / (rx_rate + tx_rate) | 流量方向不對稱（如大量上傳）|

---

### 為什麼這三個，而不是別的？

**traffic_total** — 最直觀的容量指標。
鏈路壅塞、DDoS 入侵、大型備份任務，都會先在 traffic_total 上顯現。
缺點：遮蔽了方向資訊（rx 還是 tx？）。

**error_rate** — 直接反映鏈路品質。
`rx_errs / rx_packets` 正常值應 < 0.1%；超過 1% 代表硬體問題或嚴重壅塞。
流量低時分母小，數值不穩定 → 實務上常加 `+ 1` 平滑：`errs / (packets + 1)`。

**tx_ratio** — 捕捉方向不對稱。
正常內網流量通常 rx_dominated（下載多於上傳），tx_ratio 約 0.3。
如果突然升到 0.7 以上，可能是資料洩漏（data exfiltration）或異常上傳。

---

### 特徵設計的取捨分析

| 特徵 | 優點 | 缺點 | 適用情境 |
|------|------|------|----------|
| `traffic_total` | 直觀、易解釋 | 遮蔽方向資訊 | 容量規劃、突增告警 |
| `error_rate` | 直接反映網路品質 | 流量極低時數值不穩 | 硬體故障、壅塞偵測 |
| `tx_ratio` | 捕捉方向不對稱 | 正常值因情境差異大 | 安全監控、異常上傳 |

> **工程判斷：** 哪個特徵更重要，取決於你的網路環境最常出現哪種問題。
> 這裡三個都計算是為了讓你在後面的實驗中親眼看到差異。


In [ ]:
# 計算三個複合特徵
df['traffic_total'] = df['rx_rate'] + df['tx_rate']

# error_rate = (rx_errs + tx_errs) / (rx_pkts + tx_pkts + ε)
# ε = 1e-9 防止除以零
df['error_rate'] = (
    (df['rx_errors'] + df['tx_errors']) /
    (df['rx_packets'] + df['tx_packets'] + 1e-9)
)

# tx_ratio：Transmit占比。正常家用網路通常 rx >> tx（下載多）
# 若 tx_ratio 突然升高，可能是上傳攻擊或大量備份
df['tx_ratio'] = df['tx_rate'] / (df['traffic_total'] + 1e-9)

print("複合特徵統計：")
print(df[['traffic_total', 'error_rate', 'tx_ratio']].describe().round(4))

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(df['timestamp'], df['traffic_total'] / 1e6, color='purple', linewidth=1)
axes[0].set_ylabel('MB/s')
axes[0].set_title('traffic_total（rx + tx）')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df['timestamp'], df['error_rate'] * 100, color='red', linewidth=1)
axes[1].set_ylabel('%')
axes[1].set_title('error_rate（error packet ratio %）')
axes[1].grid(True, alpha=0.3)

axes[2].plot(df['timestamp'], df['tx_ratio'], color='darkorange', linewidth=1)
axes[2].axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='50% line')
axes[2].set_ylabel('Ratio')
axes[2].set_title('tx_ratio (transmit ratio, higher = more upload traffic)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

plt.tight_layout()
plt.show()

### 特徵設計的取捨分析

| 特徵 | 優點 | 缺點 | 適用情境 |
|------|------|------|----------|
| `traffic_total` | 直觀、易解釋 | 遮蔽方向資訊 | 容量規劃、突增告警 |
| `error_rate` | 直接反映網路品質 | 流量極低時數值不穩 | 硬體故障、壅塞偵測 |
| `tx_ratio` | 捕捉方向不對稱 | 正常值因情境差異大 | 安全監控、異常上傳 |

---

## 取捨實驗：哪個特徵抓到哪種異常？

三個特徵設計來捕捉三種不同異常模式。下面用合成資料驗證這個假設。
執行後觀察：**相同的時間序列，不同特徵的反應差異有多大。**

```
Simulated scenario (3 segments of 20 samples each, injecting different anomaly types)
  A (60-80 min):   rx traffic surge 4×    →  traffic_total should spike sharply; tx_ratio unchanged
  B (100-120 min): tx traffic surge        →  tx_ratio should spike; traffic_total also rises but at different ratio
  C (140-160 min): error packet surge      →  error_rate should spike; traffic_total unchanged
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(42)
n = 200
t = pd.date_range("2024-01-01", periods=n, freq="1min")

# ── 正常基線 ──
base_rx  = 5_000_000   # 5 MB/s
base_tx  = 1_000_000   # 1 MB/s（下載遠多於上傳）
base_pkt = 10_000

rx  = base_rx  + np.random.normal(0, 200_000, n)
tx  = base_tx  + np.random.normal(0,  50_000, n)
err = np.random.poisson(2, n).astype(float)
pkt = np.random.poisson(base_pkt, n).astype(float)

# ── 注入三種異常 ──
# A：Traffic突增（rx 暴增 4×）
rx[60:80] += 4 * base_rx
# B：上傳異常（tx 暴增 8×，模擬資料外洩或備份失控）
tx[100:120] += 8 * base_tx
# C：網路品質惡化（錯誤封包從 2 封包/min 升至 1000 封包/min）
err[140:160] = np.random.poisson(1000, 20).astype(float)

df_exp = pd.DataFrame({
    "timestamp": t,
    "rx_rate": rx, "tx_rate": tx,
    "rx_errors": err, "tx_errors": err * 0.3,
    "rx_packets": pkt, "tx_packets": pkt * 0.3,
})
df_exp["traffic_total"] = df_exp["rx_rate"] + df_exp["tx_rate"]
df_exp["error_rate"]    = (
    (df_exp["rx_errors"] + df_exp["tx_errors"]) /
    (df_exp["rx_packets"] + df_exp["tx_packets"] + 1e-9)
)
df_exp["tx_ratio"] = df_exp["tx_rate"] / (df_exp["traffic_total"] + 1e-9)

# ── 正規化到 0-1 方便比較（不改變形狀，只縮放）──
def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
fig.suptitle("Feature sensitivity across three anomaly types (normalized)", fontsize=13)

colors = {"traffic_total": "purple", "error_rate": "red", "tx_ratio": "steelblue"}

for ax, feat in zip(axes, ["traffic_total", "error_rate", "tx_ratio"]):
    ax.plot(df_exp["timestamp"], minmax(df_exp[feat]),
            color=colors[feat], linewidth=1.2, label=feat)
    ax.set_ylabel(feat, fontsize=10)
    ax.axvspan(t[60],  t[79],  alpha=0.12, color="orange", label="A: RX surge")
    ax.axvspan(t[100], t[119], alpha=0.12, color="green",  label="B: TX anomaly")
    ax.axvspan(t[140], t[159], alpha=0.12, color="red",    label="C: error surge")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left", fontsize=9, ncol=4)

axes[-1].set_xlabel("Time (minutes)")
plt.tight_layout()
plt.show()

# ── 定量確認：各異常區段的特徵mean vs 基線 ──
baseline  = df_exp.iloc[:50]
seg_A     = df_exp.iloc[60:80]
seg_B     = df_exp.iloc[100:120]
seg_C     = df_exp.iloc[140:160]

ratio = pd.DataFrame({
    "traffic_total 倍率": [
        seg_A["traffic_total"].mean() / baseline["traffic_total"].mean(),
        seg_B["traffic_total"].mean() / baseline["traffic_total"].mean(),
        seg_C["traffic_total"].mean() / baseline["traffic_total"].mean(),
    ],
    "error_rate 倍率": [
        seg_A["error_rate"].mean() / (baseline["error_rate"].mean() + 1e-9),
        seg_B["error_rate"].mean() / (baseline["error_rate"].mean() + 1e-9),
        seg_C["error_rate"].mean() / (baseline["error_rate"].mean() + 1e-9),
    ],
    "tx_ratio 倍率": [
        seg_A["tx_ratio"].mean() / baseline["tx_ratio"].mean(),
        seg_B["tx_ratio"].mean() / baseline["tx_ratio"].mean(),
        seg_C["tx_ratio"].mean() / baseline["tx_ratio"].mean(),
    ],
}, index=["A: rx 突增", "B: tx 異常", "C: 錯誤激增"])

print("各異常類型相對基線的特徵放大倍率（越高代表這個特徵對此異常越敏感）：")
print(ratio.round(1).to_string())


**觀察重點：**

- 異常 A（rx 突增）：`traffic_total` 倍率最高；`tx_ratio` 反而可能下降（因為 tx 沒變）
- 異常 B（tx 上傳暴增）：`tx_ratio` 倍率最高；`traffic_total` 也上升但沒有 tx_ratio 敏感
- 異常 C（品質惡化）：只有 `error_rate` 有反應；另外兩個特徵幾乎不動

**結論：** 三個特徵不可互相替代——它們監控的是不同的維度。
在你的環境中，先判斷你最常見的異常類型，再決定哪個特徵放在偵測層的第一道閾值。

| 特徵 | 對什麼最敏感 | 容易漏掉什麼 |
|---|---|---|
| `traffic_total` | 容量事件（突增/驟降）| 品質問題（流量不大但錯誤多）|
| `error_rate` | 線路/設備品質 | 流量方向異常 |
| `tx_ratio` | 上傳異常、資料外洩、DDoS 來源 | 單純容量問題 |


---
**自訂練習** — 在動手之前，先想清楚你要偵測什麼。

---

## 🧠 設計思考：你的特徵要捕捉什麼異常？

**在填入公式前，先回答這三個問題：**

---

### Q1：你的環境最常出現哪種異常？

不同的異常在不同特徵上的「信號強度」差很多。
選錯特徵，偵測器就算演算法再好也偵測不到。

```
Anomaly type                           →  Most sensitive feature     →  Why
────────────────────────────────────────────────────────────────────────────────
Mass download / DDoS intrusion         →  traffic_total              rx spikes; total follows
Packet loss / hardware failure         →  error_rate                 errs/packets ratio jumps
Suspicious mass upload / data exfil    →  tx_ratio                   tx share rises from 0.3 to 0.7+
Link near saturation                   →  traffic_total / capacity    utilization approaches 1.0
Scheduled task spike (periodic)        →  tx_burst                   ratio vs baseline suddenly > 2
```

**實際案例 A — 資料外洩偵測（tx_ratio 的真實用途）：**
> 某金融機構內網的 tx_ratio 正常值約 0.25（主要是接收：報表下載、更新）。
> 某天下午 14:30，tx_ratio 從 0.25 攀升到 0.68 並持續 40 分鐘。
> traffic_total 同期只上升 30%，若只看 traffic_total 不會觸發告警。
> 事後調查：一台內網主機在做未授權的資料備份到外部雲端儲存。
> **tx_ratio 的方向資訊捕捉到了 traffic_total 看不到的異常。**

**實際案例 B — 為什麼 error_rate 在低流量時不可靠：**
> 凌晨 3 點，rx_packets = 5（極低流量），rx_errs = 1
> → error_rate = 1/5 = 20%，觸發告警
> 但這可能只是一個 CRC 錯誤加上幾乎沒有流量的正常夜間狀態
> **分母加 +1 可緩解，但根本解法是加入「最低流量門檻」：流量 < 1 KB/s 時忽略 error_rate**

---

### Q2：你的特徵是否需要正規化？

```
Problem                                    →  Solution              →  Specific approach
──────────────────────────────────────────────────────────────────────────────────────
Values span multiple orders of magnitude   →  log transform         log1p(error_count)
Interfaces differ 10× in traffic scale     →  relativize            value / rolling_mean
Denominator may be zero (no traffic)       →  +1 smoothing          errs / (packets + 1)
Strong periodicity, large day/night gap    →  relative deviation    (x - mean) / mean
```

---

### Q3：你的特徵對後面的偵測方法是否友善？

這個問題很多人跳過，但選錯特徵形式會讓後面三個 Lab 的努力打折：

```
For Z-score detection (Lab 02 Step 2):
  → Feature distribution should approximate a normal distribution
  → error_rate is typically heavy-tailed (near 0 most of the time, occasional spikes)
  → Solution: log-transform error_count to make the distribution more symmetric

For fixed-threshold detection (Lab 02 Step 1):
  → Feature needs a clear business-meaningful upper bound
  → traffic_total has a physical upper bound (link capacity) → suitable
  → tx_ratio has no clear "must not exceed" value → less suitable for fixed threshold

For change point detection (Lab 02 Step 4):
  → Feature must reflect a "system-level regime shift"
  → traffic_total is suitable (new service launch → permanent traffic increase)
  → error_rate is less suitable (typically a spike, not a sustained regime shift)
```

---

## ✏ 自訂複合特徵

想清楚上面三個問題後，在下方 cell 設計你的特徵。

| 特徵名稱 | 公式 | 捕捉什麼 | 適合偵測方法 |
|---------|------|---------|------------|
| `rx_drop_rate` | rx_drop / (rx_packets + 1) | 封包遺失比例 | 固定閾值、Z-score |
| `rx_tx_delta` | rx_rate − tx_rate | 收發差值 | Z-score、change point |
| `tx_burst` | tx_rate / tx_rate.rolling(12).mean() | 傳送突刺倍率 | 固定閾值（> 2.0）|
| 你的想法 | 任意公式 | 你的場景 | — |


In [ ]:
# ── 1. 為你的特徵命名（英文，無空格）─────────────────────────────────
FEATURE_NAME = 'rx_drop_rate'   # ← 改成你想要的欄位名稱

# ── 2. 填入公式（取消其中一個，或在最後寫你自己的）────────────────────────
# 選項 A：接收封包遺失率（rx_drop / packets + 1）
df[FEATURE_NAME] = (df.get('rx_drop', pd.Series(0, index=df.index))
                    / (df.get('rx_packets', pd.Series(1, index=df.index)) + 1))

# 選項 B：收發差值（正值 = 接收主導）
# df[FEATURE_NAME] = df['rx_rate'] - df['tx_rate']

# 選項 C：傳送突刺倍率（1.0 = 基線，>2 = 突刺）
# df[FEATURE_NAME] = df['tx_rate'] / (df['tx_rate'].rolling(12).mean() + 1)

# 選項 D：完全自訂
# df[FEATURE_NAME] = ...

# ── 3. 登記到自訂欄位清單（後面的彙整 cell 會自動帶入）─────────────────────
if 'CUSTOM_COLS' not in dir(): CUSTOM_COLS = []
if FEATURE_NAME not in CUSTOM_COLS: CUSTOM_COLS.append(FEATURE_NAME)
print(f"特徵 '{FEATURE_NAME}' 已加入 df")
print(df[FEATURE_NAME].describe().round(4))


In [ ]:
# 視覺化你的複合特徵 vs Raw rate
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

axes[0].plot(df['timestamp'], df['rx_rate'] / 1e6, linewidth=0.8, color='steelblue', label='rx_rate (MB/s)')
axes[0].plot(df['timestamp'], df['tx_rate'] / 1e6, linewidth=0.8, color='coral',     label='tx_rate (MB/s)')
axes[0].set_ylabel('MB/s');  axes[0].legend(fontsize=9);  axes[0].set_title('Raw rate')

axes[1].plot(df['timestamp'], df[FEATURE_NAME], linewidth=0.9, color='darkorange', label=FEATURE_NAME)
axes[1].axhline(0, color='gray', linestyle='--', linewidth=0.6)
axes[1].set_ylabel(FEATURE_NAME);  axes[1].legend(fontsize=9)
axes[1].set_title(f'Your custom feature: {FEATURE_NAME}')

plt.tight_layout();  plt.show()

# ── 結果量化 ───────────────────────────────────────────────────────────────────
print(f"\n📊 {FEATURE_NAME} 量化摘要")
print("─" * 50)
feat = df[FEATURE_NAME].dropna()
print(f"  資料筆數  ：{len(feat)}")
print(f"  均值      ：{feat.mean():.4f}")
print(f"  標準差    ：{feat.std():.4f}")
print(f"  最小 / 最大：{feat.min():.4f}  /  {feat.max():.4f}")

r_rx = feat.corr(df['rx_rate'].loc[feat.index])
r_tx = feat.corr(df['tx_rate'].loc[feat.index])
print(f"\n  與 rx_rate 相關係數：{r_rx:.4f}")
print(f"  與 tx_rate 相關係數：{r_tx:.4f}")

print("\n  🔎 自動診斷：")
if max(abs(r_rx), abs(r_tx)) > 0.97:
    print("  ⚠️  特徵與Raw rate高度相關（r > 0.97）：可能只是原始訊號的線性縮放，資訊增量有限")
else:
    print("  ✅ 特徵帶來了Raw rate以外的資訊（相關係數 < 0.97）")

z_scores = (feat - feat.mean()) / feat.std()
outliers = int((abs(z_scores) > 3).sum())
print(f"\n  |Z| > 3 的潛在異常點：{outliers} 筆 ({100*outliers/len(feat):.1f}%)")
if outliers == 0:
    print("  ⚠️  沒有 Z>3 的極端值：特徵可能過於平滑，異常的反應不夠明顯")
elif outliers / len(feat) > 0.10:
    print("  ⚠️  潛在異常比例過高（>10%）：特徵的值域可能過於集中，或噪音太大")
else:
    print("  ✅ 潛在異常比例合理，可用於後續 Z-score 偵測（Lab 02）")


---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 3：滾動統計 — 建立動態基線（Baseline）

### 為什麼需要「動態」基線？

靜態基線（全局 mean）的問題：

```
Traffic (MB/s)

  10 │              ┌──────────────── daytime peak
     │              │
   5 │ ─────────────           ────── nighttime low
     │──────────────────────────────── global mean = 7.5 MB/s
     └──────────────────────────────→ Time

  3 AM 6 MB/s vs global mean 7.5 → Z-score negative, no alert
  daytime 6 MB/s vs global mean 7.5 → Z-score negative, no alert either
  3 AM 8 MB/s vs global mean 7.5 → Z-score 0.5, no alert ← this may be anomalous!
```

**動態基線（rolling mean）的解法：**

```
  rolling_mean (12-min window) tracks the traffic:

  nighttime rolling_mean ≈ 5 MB/s
  daytime   rolling_mean ≈ 10 MB/s

  → The same deviation magnitude is detectable at any time of day
```

---

### 什麼是 rolling std？

Rolling std 衡量「過去 N 筆資料的波動程度」：
- 低 std → 流量穩定 → Z-score 分母小 → 即使小偏差也能觸發
- 高 std → 流量本身就跳動大 → Z-score 分母大 → 需要更大的偏差才觸發

這個「自動調整靈敏度」的性質，讓 Z-score 在各種流量環境下都有一定的適用性。

---

### 窗口大小的選擇指引

```
WINDOW = 12  ← 12 minutes (try 6 or 24 to observe the difference)

Window too short (< 6 min):
  Baseline pulled by recent anomalies → baseline "catches up" quickly → Z-score falls back

Window too long (> 60 min):
  Baseline too stable to reflect day/night cycle → baseline stays high during nighttime lows → sustained false alerts

Practical guidance: start at scrape_interval × 10–20 (15s × 20 = 5 min),
then tune based on observed alert rate.
```


In [ ]:
# 資料是每分鐘一筆，window 12 = 12 分鐘
# 這是後續異常偵測的基礎
WINDOW = 12  # 分鐘（可以改成 6 或 24 觀察差異）

df['rolling_mean'] = df['traffic_total'].rolling(window=WINDOW, min_periods=1).mean()
df['rolling_std']  = df['traffic_total'].rolling(window=WINDOW, min_periods=2).std().fillna(0)

# Z-score：當前值偏離移動mean幾個標準差
df['z_score'] = (df['traffic_total'] - df['rolling_mean']) / (df['rolling_std'] + 1e-9)

print(f"window大小：{WINDOW} 分鐘")
print(f"Z-score 統計：")
print(f"  mean：{df['z_score'].mean():.3f}")
print(f"  標準差：{df['z_score'].std():.3f}")
print(f"  |Z| > 3 的Ratio：{(df['z_score'].abs() > 3).mean()*100:.1f}%")

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# 上圖：原始Traffic + rolling mean ± 2σ
ax = axes[0]
ax.plot(df['timestamp'], df['traffic_total'] / 1e6, alpha=0.5, color='steelblue', linewidth=1, label='Traffic')
ax.plot(df['timestamp'], df['rolling_mean'] / 1e6, color='navy', linewidth=2, label=f'rolling mean ({WINDOW}min)')
ax.fill_between(
    df['timestamp'],
    (df['rolling_mean'] - 2 * df['rolling_std']) / 1e6,
    (df['rolling_mean'] + 2 * df['rolling_std']) / 1e6,
    alpha=0.2, color='navy', label='±2σ band'
)
ax.set_ylabel('MB/s')
ax.set_title(f'traffic_total and {WINDOW}min rolling baseline')
ax.legend()
ax.grid(True, alpha=0.3)

# 下圖：Z-score
ax2 = axes[1]
ax2.plot(df['timestamp'], df['z_score'], color='darkorange', linewidth=1, label='Z-score')
ax2.axhline(3,  color='red', linestyle='--', alpha=0.7, label='Z=±3 threshold')
ax2.axhline(-3, color='red', linestyle='--', alpha=0.7)
ax2.axhline(0,  color='gray', linestyle='-', alpha=0.3)
anomalies = df[df['z_score'].abs() > 3]
if len(anomalies) > 0:
    ax2.scatter(anomalies['timestamp'], anomalies['z_score'],
                color='red', zorder=5, s=50, label=f'anomaly points ({len(anomalies)})')
ax2.set_ylabel('Z-score')
ax2.set_title('Z-score（standard deviations from baseline）')
ax2.legend()
ax2.grid(True, alpha=0.3)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

plt.tight_layout()
plt.show()
print(f"\n偵測到 {len(anomalies)} 個 |Z-score| > 3 的點")

---
**自訂練習** — 先決定「你的正常」長什麼樣子，再選窗口大小。

---

## 🧠 設計思考：你的動態基線需要多長的記憶？

---

### Q1：你的異常事件通常持續多久？

這個問題決定了 WINDOW 的**上限**。
核心問題：如果異常持續超過 WINDOW 長度，基線會被拉偏，Z-score 會自動回落。

```
Scenario: WINDOW = 12, anomaly starts at t=50 and lasts 20 minutes

  t=50: Z = 4.1  ← alert triggered ✓
  t=55: Z = 3.8  ← still triggered
  t=62: Z = 2.1  ← baseline contaminated; Z-score falls back; alert clears ✗
  t=70: Z = 1.4  ← anomaly still present, but no longer triggered (missed alert!)

Solution: set WINDOW to at least 3× the expected anomaly duration
```

**實際案例 — DDoS 持續 45 分鐘的漏警報：**
> 某 ISP 使用 WINDOW = 10 分鐘監控骨幹鏈路。
> 一次 DDoS 攻擊持續了 45 分鐘，前 10 分鐘 Z-score 高達 6.2，觸發告警。
> 但告警在第 12 分鐘後自動解除（基線被攻擊流量污染）。
> on-call 工程師看到告警解除，誤以為問題已恢復，沒有繼續追查。
> **把 WINDOW 從 10 改為 60 分鐘後，告警持續了整個攻擊期間。**

---

### Q2：你的流量有什麼日夜週期？

```
Typical campus network traffic in a day:

  Traffic (MB/s)
  ↑
12│                 ████████
  │             ████        ████
 8│         ████                ████
  │     ████                        ████
 4│████                                  ████
  └──────────────────────────────────────────→ Time
    0:00   6:00  9:00  12:00  18:00  21:00  0:00
    night  dawn  class  lunch  after- evening lights-
                               school study   out

WINDOW = 12 min  → baseline only looks at last 12 min → catches up fast at 9 AM → class-hour peak triggers alert
WINDOW = 120 min → baseline spans peaks and troughs → daytime peak does not trigger (mean already includes peak)
WINDOW = 30 min  → balanced: can track gradual traffic increase without being pulled by brief peaks
```

**經驗法則：**
```
WINDOW minimum  = anomaly duration × 3
WINDOW maximum  = traffic cycle length ÷ 2
Reasonable WINDOW = within this range, choose the smaller value (sensitivity first) or larger value (stability first)
```

---

### Q3：用均值還是中位數做基線？

**實際案例 — 排程備份造成基線偏移：**
> 某公司每天凌晨 2:00 執行資料庫備份，持續 30 分鐘，流量是平時的 3 倍。
> WINDOW = 60 分鐘的 rolling_mean 包含了備份流量，基線被拉高 50%。
> 下一個真實的異常（非備份時段的流量突增）Z-score 偏低，差點漏掉。
>
> 換成 rolling_median 後：備份流量是 60 筆中的最高 30 筆，中位數不受影響。
> **Z-score 的靈敏度恢復，同時備份窗口的告警也減少了。**

```
rolling_mean    → all samples weighted equally → outliers (e.g. scheduled backups) bias the baseline
rolling_median  → takes the middle value         → robust to outliers; suitable for environments with periodic high-traffic events
```

---

## ✏ 自訂滾動統計基線

確認想法後，在下方 cell 填入 `WINDOW`、`BASE_COL` 和欄位命名。


In [ ]:
# ── 設定參數 ──────────────────────────────────────────────────────────────
WINDOW   = 12          # ← 窗口大小（分鐘）：試試 6, 24, 48
BASE_COL = 'rx_rate'   # ← 要計算基線的欄位：試試 'tx_rate', 'traffic_total'

# ── 為產生的欄位命名（改成對你有意義的名稱）──────────────────────────────────
MEAN_COL   = f'roll_mean_{WINDOW}m'   # 例：roll_mean_12m
STD_COL    = f'roll_std_{WINDOW}m'
ZSCORE_COL = f'zscore_{WINDOW}m'

# ── 計算 ────────────────────────────────────────────────────────────────────
df[MEAN_COL]   = df[BASE_COL].rolling(WINDOW).mean()
df[STD_COL]    = df[BASE_COL].rolling(WINDOW).std()
df[ZSCORE_COL] = (df[BASE_COL] - df[MEAN_COL]) / (df[STD_COL] + 1e-9)

# ── 可選：換成中位數基線（更抗離群值，取消下面兩行）──────────────────────────
# MEDIAN_COL = f'roll_median_{WINDOW}m'
# df[ZSCORE_COL] = (df[BASE_COL] - df[MEDIAN_COL]) / (df[STD_COL] + 1e-9)

# ── 登記 ────────────────────────────────────────────────────────────────────
if 'CUSTOM_COLS' not in dir(): CUSTOM_COLS = []
for col in [MEAN_COL, STD_COL, ZSCORE_COL]:
    if col not in CUSTOM_COLS: CUSTOM_COLS.append(col)
print(f"窗口={WINDOW}min  |  已加入：{MEAN_COL}, {STD_COL}, {ZSCORE_COL}")
print(df[ZSCORE_COL].describe().round(3))
print(f"超過 Z=3 的點：{(df[ZSCORE_COL].abs() > 3).sum()} 筆")


In [ ]:
# 視覺化你的動態基線
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

valid = df.dropna(subset=[MEAN_COL])
Z_THRESH_VIZ = 3.0

axes[0].plot(valid['timestamp'], valid[BASE_COL]/1e6,
             alpha=0.6, linewidth=0.8, color='steelblue', label=BASE_COL)
axes[0].plot(valid['timestamp'], valid[MEAN_COL]/1e6,
             linewidth=1.8, color='darkorange', label=MEAN_COL)
axes[0].fill_between(valid['timestamp'],
    (valid[MEAN_COL] - Z_THRESH_VIZ * valid[STD_COL]) / 1e6,
    (valid[MEAN_COL] + Z_THRESH_VIZ * valid[STD_COL]) / 1e6,
    alpha=0.15, color='darkorange', label=f'±{Z_THRESH_VIZ}σ')
axes[0].set_ylabel('MB/s');  axes[0].legend(fontsize=8)
axes[0].set_title(f'Your dynamic baseline ({BASE_COL}, window={WINDOW}min）')

axes[1].plot(valid['timestamp'], valid[ZSCORE_COL],
             linewidth=0.8, color='purple', alpha=0.8, label=ZSCORE_COL)
axes[1].axhline( Z_THRESH_VIZ, color='red', linestyle='--', linewidth=0.8, label='Z=±3')
axes[1].axhline(-Z_THRESH_VIZ, color='red', linestyle='--', linewidth=0.8)
axes[1].set_ylabel('Z-score');  axes[1].legend(fontsize=8)
axes[1].set_title(f'{ZSCORE_COL}')

plt.tight_layout();  plt.show()

# ── 結果量化 ───────────────────────────────────────────────────────────────────
print(f"\n📊 動態基線量化（窗口 = {WINDOW} 分鐘）")
print("─" * 50)
warmup_pct = 100 * WINDOW / len(df)
print(f"  有效資料點  ：{len(valid)} 筆（暖機丟棄 {len(df)-len(valid)} 筆 = {warmup_pct:.1f}%）")
print(f"  基線均值範圍：{valid[MEAN_COL].min()/1e6:.2f} ~ {valid[MEAN_COL].max()/1e6:.2f} MB/s")
print(f"  基線 σ 範圍 ：{valid[STD_COL].min()/1e6:.3f} ~ {valid[STD_COL].max()/1e6:.3f} MB/s")
print(f"  Z-score 峰值：{valid[ZSCORE_COL].abs().max():.2f}")

z_above3 = int((abs(valid[ZSCORE_COL]) > Z_THRESH_VIZ).sum())
alert_pct = 100 * z_above3 / len(valid)
print(f"  |Z| > 3 點數 ：{z_above3} 筆 ({alert_pct:.1f}%)")

print("\n  🔎 自動診斷：")
if warmup_pct > 30:
    print(f"  ⚠️  暖機佔比 {warmup_pct:.0f}% 偏高（>30%）：WINDOW 相對於資料集太大")
elif alert_pct > 12:
    print(f"  ⚠️  潛在異常占 {alert_pct:.1f}%（>12%）：WINDOW 可能太短，基線被異常污染")
elif alert_pct < 0.5:
    print(f"  ⚠️  潛在異常只有 {alert_pct:.2f}%（<0.5%）：WINDOW 可能太長，基線過於平滑")
else:
    print(f"  ✅ 暖機比例 {warmup_pct:.1f}%，異常比例 {alert_pct:.1f}%，窗口設定合理")

# 相較於固定基線的提升
global_mean = df[BASE_COL].mean()
global_std  = df[BASE_COL].std()
z_global    = abs((df[BASE_COL] - global_mean) / global_std)
z_local     = abs(valid[ZSCORE_COL])
print(f"\n  🔄 與全局固定基線比較：")
print(f"     全局基線 Z>3 點：{int((z_global > 3).sum())} 筆")
print(f"     動態基線 Z>3 點：{z_above3} 筆")
diff = int((z_global > 3).sum()) - z_above3
if diff > 0:
    print(f"     ✅ 動態基線消除了 {diff} 個可能的誤報（抑制了正常的周期性高峰）")
elif diff < 0:
    print(f"     ✅ 動態基線多偵測了 {-diff} 個異常（對緩慢漂移更敏感）")


---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 4：Lag 特徵 — 讓模型「記住過去」

### 什麼是 Lag 特徵？

Lag 特徵把時序的歷史值作為當前時間點的輸入特徵，讓模型能學習「記憶」結構：

```
Original series:  t=1  t=2  t=3  t=4  t=5
rx_rate:           10   12   15   13   14

lag_1 feature (shift by 1 step):
t=1: NaN  ← no previous sample
t=2: 10   ← value at t=1
t=3: 12   ← value at t=2
t=4: 15   ← value at t=3
...

Model sees: (current=13, 1-min-ago=15) → traffic is decreasing
```

### 為什麼 Lag 特徵有用？

**自相關性（Autocorrelation）：**
網路流量不是隨機的——5 分鐘前的流量和現在的流量有關聯性（正相關）。
Lag 特徵讓模型利用這個自相關性，提高偵測準確度。

```
lag_1  (1 min ago):  captures "sudden change" (lag_1 low but current high → spike)
lag_6  (6 min ago):  captures "short-term trend"
lag_12 (12 min ago): captures "medium-term periodicity" (e.g. scheduled task every 10 min)
lag_60 (1 hr ago):   captures "intra-day periodicity" (e.g. hourly report generation)
```

### 散佈圖中看什麼？

下方的 `rx_rate` vs `rx_lag_12` 散佈圖：
- 點接近對角線 → 強自相關（12 分鐘前和現在的流量高度相關）
- 點散開 → 弱自相關（流量在 12 分鐘內有大幅波動）

相關係數（r）越接近 1，lag 特徵越有預測價值。


In [ ]:
# 計算多個 lag 特徵
LAGS = [1, 3, 6, 12]  # 分鐘

for lag in LAGS:
    df[f'rx_lag_{lag}'] = df['rx_rate'].shift(lag)

lag_cols = [f'rx_lag_{lag}' for lag in LAGS]

print("Lag 特徵預覽（前 15 筆）：")
preview = df[['timestamp', 'rx_rate'] + lag_cols].head(15)
# 格式化輸出（MB/s）
for col in ['rx_rate'] + lag_cols:
    preview[col] = preview[col].apply(lambda x: f"{x/1e6:.3f}" if pd.notna(x) else 'NaN')
print(preview.to_string(index=False))

# 計算自相關性（lag 特徵與目標的相關程度）
print("\nLag 特徵與 rx_rate 的Correlation：")
for lag in LAGS:
    corr = df['rx_rate'].corr(df[f'rx_lag_{lag}'])
    bar = '█' * int(abs(corr) * 20)
    print(f"  lag_{lag:2d}min: {corr:.3f}  {bar}")

print("\n解讀：Correlation越高 → Traffic越有『記憶性』（autocorrelation）")
print("Correlation > 0.8 代表短期Traffic高度可預測")

In [ ]:
# 視覺化：rx_rate and lag_12 的散布圖（看自相關結構）
df_valid = df.dropna(subset=['rx_lag_12'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(df_valid['rx_lag_12'] / 1e6, df_valid['rx_rate'] / 1e6,
               alpha=0.3, s=20, color='steelblue')
axes[0].set_xlabel('rx_rate (lag_12, MB/s)')
axes[0].set_ylabel('rx_rate (current, MB/s)')
axes[0].set_title('Autocorrelation scatter: rx_rate vs lag_12')
axes[0].grid(True, alpha=0.3)

# Autocorrelation 折線
max_lag = min(60, len(df) // 3)
acf_vals = [df['rx_rate'].corr(df['rx_rate'].shift(i)) for i in range(1, max_lag+1)]
axes[1].bar(range(1, max_lag+1), acf_vals, color='steelblue', alpha=0.6)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].axhline(0.2, color='red', linestyle='--', alpha=0.5, label='0.2 reference line')
axes[1].set_xlabel('Lag (minutes)')
axes[1].set_ylabel('Correlation')
axes[1].set_title('Autocorrelation function (ACF)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("\n若 ACF 在大 lag 仍高 → 有長記憶性 → 適合較大的 lag 特徵")

---
**自訂練習** — 先找出你的流量週期結構，再決定 lag 長度。

---

## 🧠 設計思考：你的流量有哪些時間規律？

Lag 特徵的價值取決於自相關性。**沒有週期性的序列，lag 特徵沒有意義**。
花 5 分鐘想清楚流量的規律，比盲目試 lag 值有效得多。

---

### Q1：列出你環境中的已知週期性

```
Common periodic sources           →  Corresponding lag (min)  →  Expected autocorrelation r
────────────────────────────────────────────────────────────────────────────────
Scheduled health check every 5 min  →  lag_5                    r ≈ 0.8–0.95
Metric aggregation every 15 min     →  lag_15                   r ≈ 0.7–0.9
Hourly report generation (DB load)  →  lag_60                   r ≈ 0.5–0.8
Daily business peak (9 AM)          →  lag_1440                 r ≈ 0.6–0.9 (requires 24h data)
No pattern (random bursts)          →  any lag                  r ≈ 0.0–0.2 (lag not useful)
```

**實際案例 A — 發現隱藏的 15 分鐘週期：**
> 某電信公司的骨幹路由器流量，肉眼看起來幾乎隨機。
> 計算 lag_1 到 lag_30 的自相關係數：
>
>   lag_1: r = 0.91  ← 相鄰分鐘高度相關（流量慣性）
>   lag_15: r = 0.78 ← 驚人的 15 分鐘相關性
>   lag_30: r = 0.65 ← 30 分鐘相關性
>
> 追查發現：路由器每 15 分鐘執行一次 OSPF 路由表更新，會造成短暫流量重分配。
> **lag_15 特徵讓模型能「預期」這個規律，把它從異常偵測中排除。**

**實際案例 B — lag 特徵發現記憶體洩漏：**
> 某服務的 CPU 使用率時序有強烈的 lag_1 自相關（r = 0.98），但 lag_60 的 r 只有 0.3。
> 這代表：相鄰分鐘的 CPU 高度相關，但 1 小時後的值幾乎無關。
> 深入分析：CPU 緩慢爬升（記憶體洩漏），每 1 小時重啟後歸零。
> **lag_60 的低自相關揭露了「重啟週期」的存在。**

---

### Q2：lag 長度和「預測 horizon」的關係

```
To achieve "N-minute early warning":
  lag_N tells the model "what the value was N minutes ago"

  Example: early warning of a traffic surge 15 minutes ahead
  → requires autocorrelation r > 0.6 at lag_15 (needed for predictive value)
  → if r < 0.3, the flow 15 minutes later is barely related to the current value → 15-min early warning is not feasible

Decision tree:
  Compute r(lag_N)
  r > 0.7   → strong autocorrelation; worth using as a feature; N-min early warning feasible
  0.3–0.7   → weak autocorrelation; usable but limited effect
  < 0.3     → near-zero correlation; do not add this lag (adds noise, not signal)
```

---

### Q3：lag 太多的副作用

```
LAGS = [1, 60, 1440]  →  dropna discards the first 1440 rows (approximately 1 day)

Data volume: 360 rows (6 hours × 1 sample/min)
After dropna: 360 − 1440 = negative → all data discarded!

Rule: max(LAGS) < total rows × 0.2
With 360 rows → max(LAGS) < 72 (approximately 1.2 hours)
To use lag_1440 → need at least 7,200 rows (5 days) of data
```

---

## ✏ 自訂 Lag 特徵

先列出你環境中的週期性，計算自相關後，再決定哪些 lag 值得加入。
在下方 cell 填入 `LAG_COL` 和 `LAGS`。


In [ ]:
# ── 設定參數 ──────────────────────────────────────────────────────────────
LAG_COL = 'rx_rate'    # ← 要計算 lag 的欄位：試試 'tx_rate', 'traffic_total'
LAGS    = [1, 5, 15]   # ← lag 長度（分鐘）：試試 [1, 6, 12, 60]

# ── 欄位名稱自動由 {LAG_COL}_lag_{lag} 組成，不需另外命名 ───────────────────
lag_cols = []
for lag in LAGS:
    col = f'{LAG_COL}_lag_{lag}'
    df[col] = df[LAG_COL].shift(lag)
    lag_cols.append(col)

# ── 登記 ────────────────────────────────────────────────────────────────────
if 'CUSTOM_COLS' not in dir(): CUSTOM_COLS = []
for col in lag_cols:
    if col not in CUSTOM_COLS: CUSTOM_COLS.append(col)
print(f"目標欄位：{LAG_COL}  |  Lag：{LAGS} 分鐘")
print(f"已加入：{lag_cols}")
print(df[[LAG_COL] + lag_cols].head(8).round(0).to_string())


In [ ]:
# 視覺化：自相關散佈圖（當前值 vs lag 值）
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(4 * len(LAGS), 4))
gs  = gridspec.GridSpec(1, len(LAGS), figure=fig)

r_table = {}
for j, lag in enumerate(LAGS):
    col   = f'{LAG_COL}_lag_{lag}'
    valid = df.dropna(subset=[col])
    r     = valid[LAG_COL].corr(valid[col])
    r_table[lag] = r

    ax = fig.add_subplot(gs[j])
    ax.scatter(valid[col]/1e6, valid[LAG_COL]/1e6, alpha=0.25, s=5, color='steelblue')
    ax.set_xlabel(f'lag_{lag}min (MB/s)', fontsize=9)
    if j == 0: ax.set_ylabel('current (MB/s)', fontsize=9)
    ax.set_title(f'lag_{lag}min\nr = {r:.3f}', fontsize=10)

plt.suptitle(f'{LAG_COL} autocorrelation (r closer to 1 means the lag is more predictive)', y=1.02, fontsize=11)
plt.tight_layout();  plt.show()

# ── 結果量化 ───────────────────────────────────────────────────────────────────
print(f"\n📊 Lag 特徵自相關量化")
print("─" * 50)
max_lag = max(LAGS)
nan_loss = min(max_lag, len(df))
print(f"  max(LAGS) = {max_lag} min → dropna 移除前 {nan_loss} 筆 ({100*nan_loss/len(df):.1f}%)")
print()
print(f"  {'lag (min)':>10}  {'r':>7}  {'信號條'}  {'建議'}")
print(f"  {'─'*10}  {'─'*7}  {'─'*16}  {'─'*20}")
for lag, r in sorted(r_table.items()):
    bar   = '█' * int(abs(r) * 16)
    if abs(r) >= 0.7:
        advice = "✅ 強自相關，適合作特徵"
    elif abs(r) >= 0.35:
        advice = "⚠️  弱自相關，效果有限"
    else:
        advice = "❌ 幾乎無關，考慮移除"
    print(f"  lag_{lag:>6}min  {r:>7.4f}  {bar:<16}  {advice}")

print()
if max_lag > len(df) * 0.2:
    print(f"  ⚠️  max(LAGS)={max_lag} 超過資料長度 20%（{int(len(df)*0.2)} min）")
    print(f"     dropna 移除過多資料。若希望保留更多資料，縮小最大 lag 值")
else:
    print(f"  ✅ max(LAGS)={max_lag} 在合理範圍（< 資料長度 20%）")


---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 5：多解析度比較 — 平滑度 vs 靈敏度的取捨

### 為什麼要做多解析度分析？

同一條時序在不同時間粒度下，「看起來」截然不同：

```
Raw data (15-s resolution)     5-minute aggregate     15-minute aggregate
  ↑ Traffic                    ↑                      ↑
  │ ╱╲  ╱╲ ╱╲                  │  ╱╲                  │  ───────
  │╱  ╲╱  ╱  ╲                 │ ╱  ╲   ╲             │
  │        ╲╱  ╲               │╱    ╲───╲             │
  └──────────────→ Time         └─────────→ Time        └──────────→ Time

  High noise, fine detail        Spikes smoothed         Trend only
  Good for: second-level         Good for: alert          Good for: capacity
            anomaly detection              triggering               planning
```

### 重採樣 vs 窗口聚合

**重採樣（resample）**：把資料從高頻降到低頻，每個時段取 mean/max/sum
**滾動窗口（rolling）**：保持原始頻率，但每個點是過去 N 點的統計值

本 Step 使用重採樣，讓你直觀比較三種「視角」的差異。

### 觀察重點

執行後比較三張圖：
1. **哪些峰值在 1min 圖中明顯，但在 15min 圖中消失？**
   → 這些是短暫突刺，可能是應用程式的 burst，不是真正的流量異常
   
2. **哪些高峰在三張圖中都看得到？**
   → 這些才是持續性的流量事件，值得告警

3. **15min 圖的趨勢（是否上升？）**
   → 趨勢分析——流量是否在緩慢增長？


In [ ]:
# 重採樣到三種解析度
# 設定 timestamp 為 index 以使用 resample
df_ts = df[['timestamp', 'rx_rate']].copy()
df_ts = df_ts.set_index('timestamp')

df_1min  = df_ts.resample('1min').mean()   # 1 分鐘（原始解析度）
df_5min  = df_ts.resample('5min').mean()   # 5 分鐘（平滑）
df_15min = df_ts.resample('15min').mean()  # 15 分鐘（趨勢）

print(f"各解析度資料筆數：")
print(f"  1 分鐘：{len(df_1min)} 筆")
print(f"  5 分鐘：{len(df_5min)} 筆")
print(f" 15 分鐘：{len(df_15min)} 筆")

fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(df_1min.index, df_1min['rx_rate'] / 1e6,
        color='lightblue', linewidth=1, alpha=0.8, label='1 min (sensitive, noisy)')
ax.plot(df_5min.index, df_5min['rx_rate'] / 1e6,
        color='steelblue', linewidth=2, label='5 min (balanced)')
ax.plot(df_15min.index, df_15min['rx_rate'] / 1e6,
        color='navy', linewidth=2.5, label='15 min (clear trend, less detail)')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_ylabel('Receive rate (MB/s)')
ax.set_title('Multi-resolution comparison: same data, different resampling intervals')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 計算各解析度的峰值偵測能力
pk_1  = df_1min['rx_rate'].max()
pk_5  = df_5min['rx_rate'].max()
pk_15 = df_15min['rx_rate'].max()

print(f"\n各解析度能捕捉到的峰值：")
print(f"  1 分鐘：{pk_1/1e6:.2f} MB/s")
print(f"  5 分鐘：{pk_5/1e6:.2f} MB/s  （{pk_5/pk_1*100:.0f}% of 原始峰值）")
print(f" 15 分鐘：{pk_15/1e6:.2f} MB/s  （{pk_15/pk_1*100:.0f}% of 原始峰值）")
print("\n結論：解析度越粗 → 峰值被平均掉 → 短暫的Traffic突刺可能被遺漏")

### 多解析度取捨分析

| 解析度 | 優點 | 缺點 | 適用場景 |
|--------|------|------|----------|
| 15s ~ 1min | 最靈敏，捕捉短暫突刺 | 噪音多，誤警報多 | 秒級 SLA 保障 |
| 5min | 平衡靈敏與穩定 | 2-3 分鐘內的突刺被稀釋 | 日常運維監控 |
| 15min+ | 趨勢分析清晰 | 無法偵測短暫異常 | 容量規劃、報表 |

---
**自訂練習** — 先決定告警和報表需求，再選解析度。

---

## 🧠 設計思考：你的監控需要幾種「視角」？

解析度選擇是**最常被忽視卻影響最大**的特徵工程決策。
選錯解析度，你可能完全看不到最重要的異常。

---

### Q1：你的 on-call 團隊響應時間是多久？

響應時間決定「最有意義的解析度」：

```
Scenario: DDoS attack starts at 14:00, lasts 8 minutes, ends at 14:08

  1-minute resolution:  sees 8 anomalous data points → alert triggered ✓
  5-minute resolution:  attack diluted to 2 data points, traffic looks "only 1.6× normal" → may not trigger ✗
  15-minute resolution: attack disappears entirely in the average → completely undetectable ✗✗

If your response time is 30 minutes, an 8-minute DDoS is already over regardless.
Is "detected but unable to respond" more meaningful than "not detected"?
```

**實際案例 — 解析度盲點造成的 SLA 違規：**
> 某雲端服務提供商用 5 分鐘解析度監控 API 延遲。
> 每天上午 10:00 會有持續 90 秒的延遲突刺（約正常值的 5 倍）。
> 在 5 分鐘聚合下，這 90 秒被稀釋成「平均延遲只高 25%」→ 不觸發告警。
> 但客戶的 SLA 條款是「延遲 > 2× 正常持續 60 秒以上即違規」。
> **換成 1 分鐘解析度後，才真正捕捉到這個持續性違規。**

---

### Q2：你監控的異常最短持續多久？

```
Shortest anomaly duration to capture  →  Resolution requirement
──────────────────────────────────────────────────────────────────────
< 30 s (packet burst)                  →  second-level (15s or finer)
30 s – 3 min                           →  1 minute
3 – 15 min                             →  5 minutes
> 15 min                               →  15 minutes or coarser is sufficient

⚠ Note: finer resolution brings more noise; requires stricter deadband filtering
```

---

### Q3：不同解析度適合哪些使用場景？

**同一個事件在不同解析度下的「敘事」完全不同：**

```
Scenario: link quality begins slow degradation at 3 PM one afternoon, persists 4 hours, emergency maintenance at 1 AM

  1-minute resolution:
    → sees hundreds of small Z-score peaks, each appearing as a separate event
    → alert storm: on-call woken 30 times
    → cannot tell whether this is "sustained degradation" or "intermittent spikes"

  15-minute resolution:
    → sees a smoothly rising trend line, clearly showing "degradation from 3 PM"
    → triggers only 1 trend alert; engineer immediately sees this as a sustained problem
    → but an 8-minute DDoS is completely invisible at this resolution

  Practical strategy: overlay two resolutions
    - Fine resolution (1 min):   detects spike-type events
    - Coarse resolution (15 min): detects slow degradation trends
```

---

### 三種解析度的實際用途分工

| 解析度 | 使用場景 | 典型告警類型 |
|--------|---------|------------|
| < 1 min | 實時異常偵測、SLA 秒級監控 | 突刺、封包 burst、瞬時延遲 |
| 1–5 min | 一般維運告警 | 流量異常、Z-score 偵測 |
| 15–60 min | 趨勢告警、容量預警 | 緩慢退化、change point |
| 1 day+ | 容量規劃、報表 | 月增長率、季節性分析 |

---

## ✏ 自訂多解析度分析

結合你的響應時間和異常類型，在下方 cell 選擇 2–3 個最有意義的解析度。


In [ ]:
# ── 設定參數 ──────────────────────────────────────────────────────────────
RESOLUTIONS = ['1min', '10min', '30min']   # ← 改成你想比較的解析度：試試 ['30s', '5min', '60min']
RES_COL     = 'rx_rate'                    # ← 目標欄位：試試 'traffic_total', 'tx_rate'

# ── 計算（結果為獨立 Series，不寫入 df）────────────────────────────────────
_df_ts = df[['timestamp', RES_COL]].copy().set_index('timestamp')
resampled = {res: _df_ts[RES_COL].resample(res).mean() for res in RESOLUTIONS}

print(f"目標欄位：{RES_COL}")
for res, s in resampled.items():
    v = s.dropna()
    print(f"  {res:6s}: {len(v):4d} 點  |  mean={v.mean()/1e6:.3f} MB/s  max={v.max()/1e6:.3f} MB/s  std={v.std()/1e6:.3f} MB/s")


In [ ]:
# 視覺化：你選的multi-resolution comparison
colors = ['steelblue', 'darkorange', 'seagreen', 'crimson']
fig, axes = plt.subplots(len(RESOLUTIONS), 1,
                         figsize=(12, 3 * len(RESOLUTIONS)), sharex=False)
if len(RESOLUTIONS) == 1:
    axes = [axes]

res_stats = {}
for ax, res, color in zip(axes, RESOLUTIONS, colors):
    s = resampled[res].dropna()
    res_stats[res] = s
    ax.plot(s.index, s.values / 1e6, linewidth=1.0, color=color)
    ax.set_ylabel('MB/s', fontsize=9)
    ax.set_title(f'{res} resolution ({len(s)} points)', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'{RES_COL} — multi-resolution comparison', y=1.01, fontsize=12)
plt.tight_layout();  plt.show()

# ── 結果量化 ───────────────────────────────────────────────────────────────────
print(f"\n📊 多解析度量化比較（欄位：{RES_COL}）")
print("─" * 70)
print(f"  {'解析度':>8}  {'資料點':>6}  {'壓縮比':>6}  {'峰值 MB/s':>10}  {'變異係數 CV':>12}  {'建議用途'}")
print(f"  {'─'*8}  {'─'*6}  {'─'*6}  {'─'*10}  {'─'*12}  {'─'*20}")

base_len = len(df)
for res in RESOLUTIONS:
    s   = res_stats[res]
    cv  = s.std() / s.mean() if s.mean() != 0 else float('nan')
    ratio = base_len / len(s)
    if cv > 0.4:
        usage = "突刺偵測、即時告警"
    elif cv > 0.15:
        usage = "異常偵測、趨勢觀察"
    else:
        usage = "長期趨勢、容量規劃"
    print(f"  {res:>8}  {len(s):>6}  {ratio:>5.1f}x  {s.max()/1e6:>10.2f}  {cv:>12.4f}  {usage}")

print()
print("  📝 判讀指引：")
print("     CV（變異係數）= 標準差 / 均值")
print("     CV 高 → 保留了細節波動，適合短時間告警")
print("     CV 低 → 平滑後的趨勢訊號，適合容量規劃和長期監控")
print()
finest = min(RESOLUTIONS, key=lambda r: res_stats[r].index.freq.nanos if hasattr(res_stats[r].index, 'freq') and res_stats[r].index.freq else len(RESOLUTIONS))
if len(res_stats[finest]) < 30:
    print(f"  ⚠️  最細解析度 ({finest}) 只剩 {len(res_stats[finest])} 筆，統計效力有限")


---
**探索練習** — 修改以下 cell 中的 `WINDOW` 參數，觀察 Z-score 靈敏度的變化。

---

## 探索練習：調整滾動窗口大小

**建議操作：**
- 將 `WINDOW_EXPLORE` 從 `12` 改為 `6` → 觀察 Z-score 曲線變得更「尖銳」（更多觸發）
- 改為 `24` → 觀察 Z-score 曲線變得更「平滑」（更少觸發）
- 改為 `60` → 觀察基線非常穩定，但短暫突刺可能低於閾值

**同時調整 Z_THRESHOLD：**
- `Z_THRESHOLD = 2.0` → 非常敏感，幾乎所有波動都會觸發
- `Z_THRESHOLD = 3.0` → 標準設定，約 0.3% 的點會觸發（若資料接近常態）
- `Z_THRESHOLD = 4.0` → 只偵測極端事件

**你需要找到的「甜蜜點」：**
```
WINDOW too small → baseline contaminated by anomaly → Z-score falls back → missed alerts
WINDOW too large → cannot adapt to day/night cycle → normal daytime traffic triggers → false alerts

Z_THRESHOLD too low → too many alerts → alert fatigue
Z_THRESHOLD too high → genuine anomalies missed
```

**記錄你的觀察：**
試完不同參數後，思考：你的網路環境適合什麼組合？在下方 cell 寫下你的結論。


In [ ]:
# 🔧 修改這裡！
WINDOW_EXPLORE = 12  # 試試 6, 24, 48
Z_THRESHOLD    = 3.0  # 試試 2.0, 2.5, 4.0

df['z_explore'] = (
    (df['traffic_total'] - df['traffic_total'].rolling(WINDOW_EXPLORE, min_periods=1).mean()) /
    (df['traffic_total'].rolling(WINDOW_EXPLORE, min_periods=2).std().fillna(1) + 1e-9)
)
flags = df['z_explore'].abs() > Z_THRESHOLD
print(f"window={WINDOW_EXPLORE}min, Zthreshold={Z_THRESHOLD}：")
print(f"  異常標記數：{flags.sum()} 筆（{flags.mean()*100:.1f}%）")

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(df['timestamp'], df['z_explore'], color='darkorange', linewidth=1)
ax.axhline(Z_THRESHOLD, color='red', linestyle='--', alpha=0.6)
ax.axhline(-Z_THRESHOLD, color='red', linestyle='--', alpha=0.6)
ax.scatter(df.loc[flags, 'timestamp'], df.loc[flags, 'z_explore'],
           color='red', s=30, zorder=5, label=f'anomalies ({flags.sum()})')
ax.set_title(f'Z-score（window={WINDOW_EXPLORE}min, threshold=±{Z_THRESHOLD}）')
ax.set_ylabel('Z-score')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
**討論暫停 ▸ 全班討論** — 暫停執行，與全班分享你的觀察。

---

## ⚠ 人類驗證點 #1 — 特徵設計是你的決定

### 三個核心判斷

**判斷一：你的環境最常見的異常類型是什麼？**

回顧剛才的取捨實驗——`traffic_total`、`error_rate`、`tx_ratio` 各自對不同異常最敏感。

```
Common anomaly type → recommended primary feature

"Sudden mass download/upload"         → traffic_total (total throughput)
"Packet loss, CRC errors"             → error_rate (quality indicator)
"Suspicious mass upload behavior"     → tx_ratio (directional asymmetry)
"Slow link degradation"               → rolling_std (volatility trend) + change point (Lab 02)
"DDoS intrusion"                      → traffic_total + error_rate combined
```

**判斷二：滾動窗口大小（WINDOW）**

窗口大小決定「基線的時間記憶長度」：

| WINDOW | 基線特性 | 適合場景 |
|--------|---------|---------|
| 6 min | 追蹤快，但易被異常拉偏 | 短暫突刺偵測 |
| 12 min（預設）| 平衡靈敏與穩定 | 一般網路監控 |
| 30 min | 穩定，但對快速恢復的異常靈敏度低 | 趨勢變化偵測 |
| 60 min+ | 非常穩定，適合捕捉慢速漂移 | 容量規劃 |

**執行下方探索練習，在你的實際資料上觀察差異，再做決定。**

---

**判斷三：監控哪些介面？**

`node_network_*` 會回傳所有介面（包括 loopback、Docker bridge、VPN 虛擬介面）。
不是所有介面都需要監控——選錯介面的代價是後面所有 lab 的偵測都在看沒有意義的流量。

```
Interface naming patterns (reference):
  lo       → loopback; always exclude
  eth0/1   → wired network; typically the primary monitoring target
  en0/en1  → macOS wired/wireless; typically monitor en0
  wlan0    → wireless network
  docker0  → Docker bridge; exclude
  veth*    → Docker container virtual interface; exclude
  tun0/1   → VPN tunnel; decide whether to monitor based on requirements
```

---

### 討論問題

> 「你選了哪個特徵作為主要偵測訊號？為什麼？」

> 「在取捨實驗中，哪個異常類型（A/B/C）最難被所有特徵同時偵測到？」

> 「如果你的窗口設定為 60 分鐘，會有什麼問題？你的環境適合這個值嗎？」


In [ ]:
# 最終特徵集合 — 整理成乾淨的 DataFrame 供後續 Lab 使用
feature_cols = ['timestamp', 'device', 'rx_rate', 'tx_rate',
                'traffic_total', 'error_rate', 'tx_ratio',
                'rolling_mean', 'rolling_std',
                'rx_lag_1', 'rx_lag_3', 'rx_lag_6', 'rx_lag_12']

# 自動帶入你在上方自訂練習中定義的所有欄位
if 'CUSTOM_COLS' in dir() and CUSTOM_COLS:
    added = [col for col in CUSTOM_COLS if col in df.columns and col not in feature_cols]
    feature_cols += added
    print(f"帶入自訂欄位：{added}")

df_features = df[[c for c in feature_cols if c in df.columns]].dropna().reset_index(drop=True)
print(f"特徵 DataFrame：{df_features.shape[0]} 筆 × {df_features.shape[1]} 個欄位")
print(f"欄位：{list(df_features.columns)}")
df_features.head(3)


In [ ]:
# ── 儲存特徵供 Lab 02 使用 ─────────────────────────────────────────────────
# outputs/workshop/ 是工作坊跨 lab 的輸出目錄，所有平台（macOS/Linux/Windows）均可寫入
_out_path = PROJECT_ROOT / "outputs" / "workshop" / "workshop_lab01_features.csv"
df_features.to_csv(_out_path, index=False)
print(f"✅ 特徵已儲存：{_out_path}")
print(f"   {df_features.shape[0]} 筆 × {df_features.shape[1]} 個欄位")
print("Lab 01 完成！接下來執行 labs/workshop/02_anomaly_detection_and_alerting.ipynb")


---

## 本章小結

### 你建立了什麼

| 技術 | 解決的問題 | 輸出 |
|------|-----------|------|
| `rate()` | Counter 只增不減，無法直接比較 | 速率（bytes/sec）|
| 複合特徵 | 單一指標遺漏跨維度信號 | `traffic_total`, `error_rate`, `tx_ratio` |
| 滾動統計 | 靜態閾值無法適應流量日夜波動 | 動態基線（rolling mean ± k·std）|
| Lag 特徵 | 偵測模型看不到歷史型態 | `lag_1`, `lag_3`, `lag_6`, `lag_12` |
| 多解析度 | 單一時間粒度可能遺漏或放大噪音 | 1min / 5min / 15min 三種視角 |

---

### 特徵工程的根本限制

特徵工程不能解決「你不知道要監控什麼」的問題。
如果你沒有先想清楚「哪種異常對業務最有影響」，選出的特徵可能捕捉不到最重要的信號。

這個問題沒有演算法可以自動回答——它需要你的領域知識。這是「⚠ 人類驗證點 #1」的核心。

---

### 特徵工程 → 異常偵測的橋接

剛才輸出的特徵 DataFrame 包含：

```python
['timestamp', 'device',
 'rx_rate', 'tx_rate',                          ← raw rates
 'traffic_total', 'error_rate', 'tx_ratio',     ← composite features
 'rolling_mean', 'rolling_std',                 ← dynamic baseline
 'rx_lag_1', 'rx_lag_3', ...]                   ← autocorrelation features
```

**Lab 02 會用這個 DataFrame 做什麼？**

1. 用 `traffic_total` 和 `rolling_mean/std` 計算 Z-score → 偵測突刺
2. 用雙移動均線發散 → 偵測狀態切換（change point）
3. 輸出結構化事件清單 → 供 Lab 08 的 Agentic AI 消費

你在這裡選的 `WINDOW` 和目標介面，會直接影響 Lab 02 的偵測靈敏度。
如果你改了 `WINDOW`，重新執行本 lab 最後一個 code cell 重新輸出 features，然後再進入 Lab 02。


## Optional: update Grafana with this result

Notebook 會顯示 feature engineering 結果。若想在 Grafana 看同一批 features，保持 `python infra/python_results_exporter.py` 執行，然後複製：

```bash
cp outputs/workshop/workshop_lab01_features.csv outputs/prometheus-dropzone/current_results.csv
```

Grafana 可查 `aiops_python_result`，再用 `column` label 選擇 feature。完整流程見 `labs/getting-started/05-prometheus-dropzone.md`。
